# AI Ticket Triage System — Google Colab

English, day-wise project notebook. Run cells from top to bottom. It creates a synthetic dataset, shows a preview and evaluation metrics, then opens the Streamlit dashboard. The dashboard URL is temporary and lasts only while this Colab runtime stays active.

## Setup — install project dependencies
This notebook writes the project source files below, so the code is visible in these cells.

In [ ]:
!pip -q install streamlit pandas scikit-learn
from pathlib import Path
Path("/content/ai_ticket_triage_system/src").mkdir(parents=True, exist_ok=True)
Path("/content/ai_ticket_triage_system/data").mkdir(parents=True, exist_ok=True)
%cd /content/ai_ticket_triage_system

## Days 3–12 — project logic
Text cleaning, category/priority prediction, sentiment labels, routing, semantic retrieval and response drafting.

In [ ]:
%%writefile src/ticket_triage.py
import re
from pathlib import Path
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,f1_score

CATEGORIES=["Billing","Login","Technical Issue","Refund","Feature Request"]
TEAMS={"Billing":"Finance Support","Login":"Account Support","Technical Issue":"Technical Support","Refund":"Finance Support","Feature Request":"Product Team"}
EXAMPLES={"Billing":["I was charged twice for my subscription","My invoice has the wrong amount","Please explain this unexpected payment","My card payment failed","I need a receipt for my bill"],"Login":["I cannot log in to my account","Password reset link is not working","I am locked out of my account","Verification code never arrived","My account login keeps failing"],"Technical Issue":["The app crashes when I upload a file","Dashboard is loading very slowly","I see an error saving settings","The website stopped working","Notifications are not appearing"],"Refund":["Please refund my recent purchase","I cancelled but did not get my money back","I want a refund for a duplicate order","Return the payment to my card","My refund is overdue"],"Feature Request":["Please add dark mode","Please export reports as PDF","Can you add team sharing","Add calendar reminders","Please support more languages"]}
def clean_text(text):
 text=re.sub(r"https?://\S+|www\.\S+"," URL ",str(text).lower()); return re.sub(r"\s+"," ",re.sub(r"[^a-z0-9\s']"," ",text)).strip()
def classify_category(text):
 t=clean_text(text); rules={"Refund":["refund","money back","return payment"],"Billing":["bill","invoice","charged","payment","subscription","receipt"],"Login":["login","log in","password","verification","locked out"],"Feature Request":["add ","feature","suggest","dark mode","export"],"Technical Issue":["crash","error","slow","loading","stopped working","bug"]}
 for label,terms in rules.items():
  if any(x in t for x in terms): return label
 return "Technical Issue"
def predict_priority(text):
 t=clean_text(text);
 for label,terms in [("Urgent",["security breach","account compromised","urgent outage","cannot access any account"]),("High",["production down","business stopped","multiple users","charged twice","deadline today"]),("Medium",["not working","incorrect","delayed","cannot","failed"]),("Low",["suggest","would like","question","feature"])]:
  if any(x in t for x in terms): return label
 return "Medium"
def analyze_sentiment(text):
 t=clean_text(text); pos=sum(x in t for x in ["thanks","thank you","great","helpful","love","excellent","happy"]); neg=sum(x in t for x in ["angry","terrible","broken","frustrated","disappointed","hate","worst","upset"]); return "Positive" if pos>neg else "Negative" if neg>pos else "Neutral"
def route_team(category): return TEAMS.get(category,"General Support")
def generate_synthetic_tickets(n_per_category=40,seed=42):
 import random
 from datetime import datetime,timedelta
 rng=random.Random(seed); rows=[]; now=datetime.now(); tid=1001
 for cat,examples in EXAMPLES.items():
  for i in range(n_per_category):
   msg=examples[i%len(examples)]+rng.choice([""," Thanks for your help."," I am frustrated."," This is disappointing."]); rows.append({"ticket_id":f"T{tid}","customer_message":msg,"date":(now-timedelta(days=rng.randint(0,60))).date().isoformat(),"customer_name":f"Customer {rng.randint(1,80)}","customer_email":f"customer{rng.randint(1,80)}@example.com","status":rng.choice(["Open","Open","In Progress","Resolved"]),"category":cat,"priority":predict_priority(msg),"sentiment":analyze_sentiment(msg),"assigned_team":route_team(cat)}); tid+=1
 return pd.DataFrame(rows)
def train_models(df):
 models={}
 for target in ["category","priority"]:
  m=Pipeline([("tfidf",TfidfVectorizer(ngram_range=(1,2))), ("clf",LogisticRegression(max_iter=1000,random_state=42))]); m.fit(df.customer_message.astype(str),df[target].astype(str)); models[target]=m
 return models
def evaluate_model(df,target):
 x1,x2,y1,y2=train_test_split(df.customer_message.astype(str),df[target].astype(str),test_size=.25,random_state=42,stratify=df[target]); m=Pipeline([("tfidf",TfidfVectorizer(ngram_range=(1,2))), ("clf",LogisticRegression(max_iter=1000,random_state=42))]); m.fit(x1,y1); p=m.predict(x2); return {"accuracy":accuracy_score(y2,p),"f1_macro":f1_score(y2,p,average="macro"),"model":m}
def retrieve_context(query,path="data/knowledge_base.txt",top_k=3):
 p=Path(path)
 if not p.is_file(): return []
 chunks=[x.strip() for x in p.read_text(encoding="utf-8").split("\n\n") if x.strip()]
 if not chunks:return []
 v=TfidfVectorizer(ngram_range=(1,2)); mat=v.fit_transform(chunks); scores=(mat@v.transform([query]).T).toarray().ravel(); return [chunks[i] for i in scores.argsort()[::-1][:top_k] if scores[i]>0]
def draft_response(message,category,priority,context):
 body=("Based on our support guide: "+context[0]) if context else f"We classified this as {category.lower()} and routed it to {route_team(category)}. A support specialist will review it."
 if priority=="Urgent":body+=" We marked this ticket urgent for prompt review."
 return "Hello, thank you for contacting support. "+body+" Please reply here if you have more details."


## Days 1–2 — create and preview synthetic ticket dataset

In [ ]:
%%writefile generate_data.py
from pathlib import Path
from src.ticket_triage import generate_synthetic_tickets
Path("data").mkdir(exist_ok=True)
generate_synthetic_tickets().to_csv("data/tickets.csv",index=False)
print("Created data/tickets.csv")


In [ ]:
!python generate_data.py
import pandas as pd
tickets = pd.read_csv("data/tickets.csv")
print("Synthetic dataset rows:", len(tickets))
display(tickets.head(10))

## Days 5–12 — Streamlit ticket dashboard
Ticket submission, filters, queue dashboard, labels, team routing and knowledge search.

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
from pathlib import Path
from src.ticket_triage import generate_synthetic_tickets,classify_category,predict_priority,analyze_sentiment,route_team,retrieve_context,draft_response
st.set_page_config(page_title="AI Ticket Triage",page_icon="🎫",layout="wide"); st.title("AI Ticket Triage System"); st.caption("Educational demo using synthetic data and transparent NLP baselines.")
data=Path("data/tickets.csv"); kb="data/knowledge_base.txt"
if not data.exists(): data.parent.mkdir(exist_ok=True); generate_synthetic_tickets().to_csv(data,index=False)
df=pd.read_csv(data)
with st.sidebar:
 cats=st.multiselect("Category",sorted(df.category.unique()),default=sorted(df.category.unique())); pris=st.multiselect("Priority",["Urgent","High","Medium","Low"],default=["Urgent","High","Medium","Low"])
f=df[df.category.isin(cats)&df.priority.isin(pris)]; a,b,c,d=st.columns(4); a.metric("Tickets",len(f)); b.metric("Open",int((f.status=="Open").sum())); c.metric("Urgent",int((f.priority=="Urgent").sum())); d.metric("Negative sentiment",int((f.sentiment=="Negative").sum()))
t1,t2,t3=st.tabs(["Ticket dashboard","Submit a ticket","Knowledge search"])
with t1:
 st.dataframe(f.sort_values("date",ascending=False),use_container_width=True,hide_index=True); x,y=st.columns(2); x.bar_chart(f.category.value_counts()); y.bar_chart(f.priority.value_counts())
with t2:
 with st.form("ticket"):
  name=st.text_input("Customer name"); email=st.text_input("Customer email"); msg=st.text_area("Customer message"); submit=st.form_submit_button("Analyze and submit")
 if submit:
  if not msg.strip():st.error("Enter a message first.")
  else:
   cat=classify_category(msg); pri=predict_priority(msg); sent=analyze_sentiment(msg); team=route_team(cat); ctx=retrieve_context(msg,kb); response=draft_response(msg,cat,pri,ctx); row={"ticket_id":f"T{1001+len(df)}","customer_message":msg,"date":pd.Timestamp.now().date().isoformat(),"customer_name":name or "Guest","customer_email":email or "Not provided","status":"Open","category":cat,"priority":pri,"sentiment":sent,"assigned_team":team}; pd.concat([df,pd.DataFrame([row])],ignore_index=True).to_csv(data,index=False); st.success(f"Ticket created; routed to {team}."); st.write({"Ticket ID":row["ticket_id"],"Category":cat,"Priority":pri,"Sentiment":sent,"Team":team}); st.markdown("**Suggested response**"); st.write(response)
with t3:
 q=st.text_input("Search support guidance")
 if q:
  hits=retrieve_context(q,kb); [st.write(f"Result {i}: {h}") for i,h in enumerate(hits,1)] if hits else st.info("No matching article found.")


## Day 13 — evaluation
Reports held-out category and priority accuracy, macro F1, and routing accuracy on synthetic data.

In [ ]:
%%writefile evaluate.py
import pandas as pd
from src.ticket_triage import evaluate_model,TEAMS
df=pd.read_csv("data/tickets.csv")
for target in ["category","priority"]:
 r=evaluate_model(df,target); print(f"{target}: accuracy={r['accuracy']:.3f}, macro F1={r['f1_macro']:.3f}")
print("Routing accuracy:", (df.assigned_team==df.category.map(TEAMS)).mean())


In [ ]:
!python evaluate.py

## Start the Streamlit dashboard
Start the app on a Colab port. The CORS/XSRF flags are included for this temporary Colab demo proxy. Do not use real customer details in a public demo.

In [ ]:
!python -m streamlit run app.py --server.port 8502 --server.address 0.0.0.0 --server.enableCORS=false --server.enableXsrfProtection=false > /content/streamlit.log 2>&1 &

In [ ]:
from google.colab import output
output.serve_kernel_port_as_iframe(8502, height=800)

## Day 14 — demo and documentation
Submit one billing ticket, one login ticket, and one urgent technical issue. Explain that the dataset is synthetic, priority and sentiment are rule-based baselines, semantic retrieval uses TF-IDF, and the response is a grounded template rather than an LLM response.